# MuseumSCAT: fine-tune doctr's Danish-vocab recognizer

Before running: attach the private dataset `museumscat-danish-recognizer-crops` as input
(Notebook Settings → Input → search for it → Add), and under Settings enable
**Accelerator = GPU T4 x2 (or P100)** and **Internet = On**.

**Two-stage recipe, v2** (see `docs/superpowers/plans/2026-07-24-danish-recognizer-finetune.md`,
"Fine-tune result" section for the full history): the first frozen-backbone attempt
(20 epochs) left the head undertrained (every date came back MISSING). A first
Stage 1/2 attempt (150/50 epoch ceilings) fixed that but finished in about a minute —
meaning early stopping tripped almost immediately, likely on validation-loss noise
from only ~4-7 batches per epoch, not real convergence.

This version is budgeted for roughly an hour of GPU time: higher epoch ceilings
(600/400), much more patient early stopping (60/50), a smaller batch size (16, more
gradient updates per epoch on this small dataset), AdamW + weight decay in Stage 2
(the riskier full-backbone unfreeze) for regularization, and `--amp` to keep it fast
on a T4's tensor cores.

In [ ]:
!pip install -q "python-doctr[torch]==1.0.1" matplotlib
!mkdir -p doctr_recognition
!curl -s -o doctr_recognition/train.py "https://raw.githubusercontent.com/mindee/doctr/v1.0.1/references/recognition/train.py"
!curl -s -o doctr_recognition/utils.py "https://raw.githubusercontent.com/mindee/doctr/v1.0.1/references/recognition/utils.py"

In [ ]:
import os

INPUT_DIR = "/kaggle/input/datasets/matiaspoullain/museumscat-danish-recognizer-crops"
TRAIN_PATH = os.path.join(INPUT_DIR, "train")
VAL_PATH = os.path.join(INPUT_DIR, "val")
assert os.path.isfile(os.path.join(TRAIN_PATH, "labels.json"))
assert os.path.isfile(os.path.join(VAL_PATH, "labels.json"))

In [ ]:
!cd doctr_recognition && python train.py crnn_vgg16_bn \
    --vocab danish --pretrained --freeze-backbone --amp \
    --train_path {TRAIN_PATH} --val_path {VAL_PATH} \
    --epochs 600 --batch_size 16 --lr 0.001 \
    --early-stop --early-stop-epochs 60 --early-stop-delta 0.0005 \
    --output_dir /kaggle/working --name danish_crnn_vgg16_bn_stage1

In [ ]:
import os

assert os.path.isfile("/kaggle/working/danish_crnn_vgg16_bn_stage1.pt")
print("stage 1 checkpoint ready: /kaggle/working/danish_crnn_vgg16_bn_stage1.pt")

In [ ]:
!cd doctr_recognition && python train.py crnn_vgg16_bn \
    --vocab danish --resume /kaggle/working/danish_crnn_vgg16_bn_stage1.pt --amp \
    --train_path {TRAIN_PATH} --val_path {VAL_PATH} \
    --epochs 400 --batch_size 16 --lr 0.0001 --optim adamw --wd 0.0001 \
    --early-stop --early-stop-epochs 50 --early-stop-delta 0.0005 \
    --output_dir /kaggle/working --name danish_crnn_vgg16_bn_stage2

In [ ]:
import os

assert os.path.isfile("/kaggle/working/danish_crnn_vgg16_bn_stage2.pt")
print("final checkpoint ready to download: /kaggle/working/danish_crnn_vgg16_bn_stage2.pt")